# Miles de opciones, tiempo desigual

**Autor: Roberto Cantero · Nebula · EDA**

## Mi pregunta y mi medida

Quiero medir el reparto del tiempo observado entre enero y junio de 2026. Ordeno por horas de mayor a menor; sumo hasta alcanzar al menos el 50 % del total. En los empates uso el título en orden alfabético para obtener un resultado reproducible. Conservo las horas de las bolsas de otros contenidos en el total, pero no las incluyo como títulos. No hago una predicción.

In [1]:
# cargo mis datos preparados y compruebo el total
import csv
import json
from pathlib import Path
from preparar_datos import resumir
raiz = Path.cwd()
with (raiz/'netflix_2026_s1.csv').open(encoding='utf-8', newline='') as archivo:
    registros = list(csv.DictReader(archivo))
for r in registros:
    r['horas'] = int(r['horas'])
    r['visualizaciones'] = int(r['visualizaciones']) if r['visualizaciones'] else None
resultados = {g:resumir([r for r in registros if g=='conjunto' or r['formato']==g]) for g in ['conjunto','series','peliculas']}
for grupo, resumen in resultados.items():
    print(grupo, {k:v for k,v in resumen.items() if k!='top'})

conjunto {'registros': 17370, 'total': 97664500000, 'horas_otros': 859900000, 'mitad': 775, 'porcentaje_titulos_mitad': 4.461715601611975, 'cuotas': {'10': 4.6438572869364, '100': 18.905026903327208, '1000': 55.29696051277588}}
series {'registros': 8193, 'total': 74354300000, 'horas_otros': 757000000, 'mitad': 473, 'porcentaje_titulos_mitad': 5.773221042353229, 'cuotas': {'10': 6.099714475154766, '100': 23.583706658525465, '1000': 67.08556196480903}}
peliculas {'registros': 9177, 'total': 23310200000, 'horas_otros': 102900000, 'mitad': 468, 'porcentaje_titulos_mitad': 5.099705786204642, 'cuotas': {'10': 7.824042693756382, '100': 24.473406491578796, '1000': 67.40825904539643}}


## Lo que encuentro

Obtengo 775 entradas para reunir la mitad de las horas mundiales. Son el 4,46 % de las 17.370 entradas identificadas, no del catálogo mundial completo. Las diez primeras suman el 4,64 % y las cien primeras el 18,91 %. Una parte reducida del listado concentra mucho tiempo, pero diez éxitos no explican la mayor parte del consumo.

In [2]:
# verifico que mi punto de corte alcanza la mitad y el anterior no
for grupo, resumen in resultados.items():
    filas = [r for r in registros if grupo=='conjunto' or r['formato']==grupo]
    orden = sorted([r for r in filas if r['titulo'] not in {'Other Movies','Other Shows'}], key=lambda r:(-r['horas'],r['titulo']))
    n = resumen['mitad']
    total = sum(r['horas'] for r in filas)
    assert sum(r['horas'] for r in orden[:n-1]) < total/2 <= sum(r['horas'] for r in orden[:n])
    print(grupo, 'antes:', round(sum(r['horas'] for r in orden[:n-1])/total*100,5), 'después:', round(sum(r['horas'] for r in orden[:n])/total*100,5))
    print('Primer título por horas:', orden[0]['titulo'], orden[0]['horas'])

conjunto antes: 49.97957 después: 50.00599
Primer título por horas: Bridgerton: Season 4 889800000
series antes: 49.95609 después: 50.00289
Primer título por horas: Bridgerton: Season 4 889800000
peliculas antes: 49.96396 después: 50.00858
Primer título por horas: War Machine 266800000


## Mi comparación entre formatos

Necesito 468 películas para reunir la mitad del tiempo de películas, y 473 temporadas u otros programas para el grupo de series. Las unidades no tienen la misma duración. No afirmo que una temporada con más horas tenga más espectadores únicos que una película.

In [3]:
# comparo mis horas con las visualizaciones equivalentes publicadas
sensibilidad = {}
for grupo in ['series','peliculas']:
    filas = [r for r in registros if r['formato']==grupo and r['visualizaciones'] is not None]
    sensibilidad[grupo] = resumir(filas, 'visualizaciones')
    print(grupo, 'mitad por visualizaciones:', sensibilidad[grupo]['mitad'], 'sobre', sensibilidad[grupo]['registros'], 'entradas identificadas')
    print('Primera entrada:', sensibilidad[grupo]['top'][0])
print('Esta comparación excluye las bolsas de otros contenidos porque no publican visualizaciones. No uso el mismo denominador que en horas.')

series mitad por visualizaciones: 567 sobre 8193 entradas identificadas
Primera entrada: {'titulo': 'His & Hers: Limited Series', 'formato': 'series', 'visualizaciones': 104000000}
peliculas mitad por visualizaciones: 474 sobre 9177 entradas identificadas
Primera entrada: {'titulo': 'War Machine', 'formato': 'peliculas', 'visualizaciones': 146900000}
Esta comparación excluye las bolsas de otros contenidos porque no publican visualizaciones. No uso el mismo denominador que en horas.


## Mi sensibilidad y mis límites

Las visualizaciones equivalentes son horas divididas por duración: no son usuarios distintos ni sesiones completas observadas. Al cambiar de medida necesito 567 entradas de series y 474 películas para la mitad de sus respectivas visualizaciones identificadas. No presento estas cifras como directamente intercambiables con los cortes por horas, pues las bolsas carecen de visualizaciones.

El resultado describe seis meses con distintas fechas de estreno y territorios. No mide gustos individuales, calidad, beneficios o el efecto causal del algoritmo. No puedo inferir que cada usuario elija siempre lo mismo. Netflix publica los datos y no dispongo de una auditoría independiente. Los redondeos pueden alterar cortes cercanos; el resultado de 775 corresponde al archivo conservado.

## Mi conclusión

Encuentro concentración en una parte pequeña del listado, pero un top 10 explica poco del tiempo total. Distingo ambas afirmaciones para evitar convertir el resultado en una historia sobre diez éxitos que no sostienen los datos.

In [4]:
# contrasto mi cálculo independiente con el archivo que usa el artículo
publicados = json.loads((raiz/'resultados.json').read_text(encoding='utf-8'))
assert all(resultados[g] == publicados[g] for g in resultados)
assert all(sensibilidad[g] == publicados['sensibilidad_visualizaciones'][g] for g in sensibilidad)
print('He reproducido los resultados del artículo a partir del CSV.')

He reproducido los resultados del artículo a partir del CSV.


## Mis fuentes

Uso netflix_2026_s1.xlsx y su CSV preparado. Conservo la procedencia, las notas y la huella del archivo en fuentes_metodologia.md y resultados.json. Puedo repetir la preparación con `python preparar_datos.py` antes de ejecutar estos dos cuadernos en orden.